# BP8 Gate 2 — Data Verification & Gold-Table Aggregation

## Why this notebook exists

BP8 Gate 1 (Business Understanding) already ran for real and named its own next step
explicitly: *"Proceed to BP8 Gate 2 (Data Verification & Gold-Table Aggregation) next."*
BP8 is a cross-BP Gold-layer aggregation and Power BI reporting layer — per BP8's own
`policy.json`, *"never a ninth modeling problem - no target, no classifier, no GenAI call."*
This notebook does three things, and only three things:

1. **Re-checks live** (not trusts Gate1's snapshot) whether each of BP1-BP7 has reached its
   own real Gate6, using a corrected multi-signal check.
2. **Builds real Gold/semantic tables** for whichever upstream BPs are genuinely ready today,
   writing them to `powerbi/gold_tables/` as Parquet.
3. **Writes a Gate2 artifact manifest** and a flat Gate2 config block, disclosing every
   correction versus Gate1's own snapshot — this project's standing convention is to disclose
   corrections, never silently apply or silently ignore them.

## The corrected `gate6_reached` check, and why Gate1's own check was insufficient

Gate1's own live-check computed `gate6_reached` as `bool(status) and "gate6" in
status.lower()`. That happens to work for bp1-bp4, whose real `status:` strings literally
gain a `"gate6"` substring once their real Gate6 completes. It is structurally broken for
bp5, bp6 and bp7: none of their `status:` fields ever grow a `"gate6"` substring, even after
their real Gate6 completes, because Gate6 deliberately never writes to `status:` for bp6/bp7,
and bp5's own Gate6 likewise never appends to `status:`. This notebook imports a corrected
`check_gate6_reached()` from `src/features/bp8_gold_table_builders.py` that adds two more real,
independent signals: a flat top-level `gate6_generated_at_utc` key (bp5's and bp7's real
convention), and a nested per-bp gate6 dict carrying its own `generated_at_utc` field
(bp1's `gate6_governance`, bp6's `gate6_productization_monitoring_governance`). Any one of the
three signals is sufficient. Run live against this bp's real config, all 7 upstream BPs now
resolve to `gate6_reached=True` — a real, disclosed change from Gate1's own snapshot, which
(using its buggy substring-only check) recorded bp5/bp6/bp7 as `gate6_reached=False`.

## The `review_priority_tier` source correction

Gate1's own `policy.json` (`kpi_category_scope.product_opportunity_flags.source_table`) names
`cfpb_issue_cluster_summary_gold.parquet` as the source of `review_priority_tier`. That
parquet's real columns (verified live) are: `Company, Product, Sub-product, Issue, Sub-issue,
n_complaints_total, first_complaint_date, last_complaint_date, n_active_months,
avg_response_lag_days, banking77_coverage_fraction, is_recurring_cluster` — there is no
`review_priority_tier` column in it. The real source is BP4's own decision-artifact index,
`models/bp4_customer_journey_analytics/bp4_decision_artifact_index.parquet`, joinable to the
same table at the same grain, `(Company, Product, Sub-product, Issue, Sub-issue)`. This
notebook's Section 11 structural checks live-prove both halves of this correction: that
`review_priority_tier` is genuinely absent from `cfpb_issue_cluster_summary_gold.parquet`, and
genuinely present in the written `bp8_gold_product_opportunity_flags.parquet`.

## The BP5 upgrade from Gate1-deferred to Gate2-ready

Gate1's own snapshot recorded `root_cause_driver_kpis` as **DEFERRED**, because its buggy
gate6 check read bp5 as not-yet-Gate6. Live re-check today shows bp5's real Gate6 output
(`gate6_generated_at_utc: 2026-09-24T11:15:03.634508+00:00` in bp5's own config) — a real,
current fact Gate1's own check could not see — together with a real Gold table that did not
exist at Gate1's run time, `cfpb_root_cause_driver_gold.parquet`, and BP5's real Gate5
field-driver-ranking JSON artifacts. `root_cause_driver_kpis` is therefore promoted to
**READY** in this Gate2, and its two Gold tables
(`bp8_gold_root_cause_outcome_trends.parquet`, `bp8_gold_root_cause_field_driver_ranking.parquet`)
are built below. This is the headline correction of this Gate2 run.

## The deliberate, conservative BP7 deferral

BP7's real Gate6 is also now done (`gate6_generated_at_utc:
2026-09-25T06:49:02.757522+00:00`), correcting Gate1's own `gate6_reached=False` snapshot for
bp7. Despite that, `decision_engine_kpis` **stays DEFERRED** in this Gate2 — the same verdict
Gate1 reached, but for an updated, more precise reason: BP7 has no
`data/processed/*_gold.parquet` final decision-output table. It has only a Gate2 input/context
Gold parquet (`cfpb_decision_engine_context_gold.parquet`) that lacks BP7's own final
`priority_score`/`recommended_action` fields. Two small real Gate5 aggregate CSVs
(`gate5_recommended_action_breakdown.csv`, `gate5_bp4_tier_intervention_crosstab.csv`) exist
under BP7's own artifacts folder and are real and governed — this notebook records their real
existence in the manifest as an informational, non-aggregated observation
(`bp7_observed_not_yet_aggregated`), but deliberately does **not** build a Gold table from them
this Gate2. The 543MB `gate5_full_population_decision_records.csv` is never opened, staged, or
referenced anywhere in this notebook — a standing project rule, mirrored by BP7's own Gate7
rollup.

## BP6 stays deferred, cleanly

BP6's real Gate6 is also now done, but BP6 has no Gold-layer parquet anywhere under
`data/processed/`, and its own `notebooks/bp6_genai_resolution_assistant/artifacts/` folder
contains only PII-screening, retrieval-strategy, explainability-trace and
human-review-pending artifacts — no quantifiable governed outcome table at all.
`genai_resolution_kpis` stays cleanly **DEFERRED**; there is no small aggregate artifact worth
flagging for BP6 the way there is for BP7.

## Standing rules this notebook follows

- **Execution boundary**: this notebook is written to be run by the user, in their own
  environment; it is never executed by Claude.
- **Zero fabrication**: every column name, file path, and real value referenced below comes
  from live-verified real data; nothing is invented, and no financial-impact or
  illustrative/assumption-based figure is ever produced.
- **WARP**: `configure_performance()` is called immediately after project-root resolution,
  before any heavy import, per project convention.
- **HYPER**: all Gold-table-build logic lives in the shared, import-only
  `src/features/bp8_gold_table_builders.py` module; this notebook only orchestrates calls into
  it.
- **Idempotent config writes**: the Gate2 config block is written via the project's existing
  `write_gate_block()` helper (`src/utils/bp1_config_sync.py`), which replaces the block in
  place on re-run rather than duplicating it. `write_front_matter()` is never called here — it
  is owned exclusively by Gate1.
- **No re-derivation of upstream logic**: BP8 never recomputes an upstream BP's own
  classification/severity/tier logic, and never reads raw CFPB/BANKING77 row-level narrative
  text or a demographic-adjacent field (`Tags`, `ZIP code`) as an output dimension.

## Real outputs of this notebook

- `powerbi/gold_tables/bp8_gold_friction_trends.parquet`
- `powerbi/gold_tables/bp8_gold_escalation_trends.parquet`
- `powerbi/gold_tables/bp8_gold_product_opportunity_flags.parquet`
- `powerbi/gold_tables/bp8_gold_customer_intent_taxonomy_trends.parquet`
- `powerbi/gold_tables/bp8_gold_customer_intent_banking77_categories.parquet`
- `powerbi/gold_tables/bp8_gold_root_cause_outcome_trends.parquet`
- `powerbi/gold_tables/bp8_gold_root_cause_field_driver_ranking.parquet`
- `notebooks/bp8_executive_product_analytics/artifacts/gate2_gold_table_manifest.json`
- An appended/updated Gate2 block in `configs/bp8_executive_product_analytics.yaml`

## Prerequisites

- BP8 Gate1 has already run for real (`configs/bp8_executive_product_analytics.yaml` already
  exists with Gate1-shaped front matter — this notebook never creates that file).
- `src/utils/performance_setup.py` and `src/utils/bp1_config_sync.py` already exist in the real
  repo and are imported unmodified.
- All 7 upstream BP config files exist under `configs/`.
- All 6 required `data/processed/*.parquet` Gold-layer inputs and the
  `models/bp4_customer_journey_analytics/bp4_decision_artifact_index.parquet` file exist.
- BP5's two real Gate5 field-driver-ranking JSON artifacts exist under
  `notebooks/bp5_root_cause_driver_analytics/artifacts/`.
- `powerbi/gold_tables/` already exists (confirmed by Gate1's own live check).

## What happens if a structural check fails

Every structural check in Section 11 below is a real `assert`, not a print-only check: a
failure raises `AssertionError` immediately and stops the notebook, so a partially-correct
Gate2 output is never silently accepted or partially written to the manifest/config as if it
had succeeded.


In [ ]:
# ============================================================================================
# SECTION 1 — Project root resolution (verbatim project convention; do not modify)
# ============================================================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the "
        "project tree (expected at notebooks/bp8_executive_product_analytics/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# ============================================================================================
# SECTION 2 — WARP performance setup (call immediately after project-root resolution, before
# any heavy import)
# ============================================================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================================================
# SECTION 3 — Heavy imports (post sys.path insert) + flush-forcing print override
# ============================================================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import polars as pl  # noqa: E402
import yaml  # noqa: E402

from features.bp8_gold_table_builders import (  # noqa: E402
    UPSTREAM_BPS,
    build_customer_intent_banking77_categories_gold,
    build_customer_intent_taxonomy_trends_gold,
    build_escalation_trends_gold,
    build_friction_trends_gold,
    build_product_opportunity_flags_gold,
    build_root_cause_field_driver_ranking_gold,
    build_root_cause_outcome_trends_gold,
    check_gate6_reached,
    gold_table_manifest,
)
from utils.bp1_config_sync import write_gate_block  # noqa: E402

print = functools.partial(builtins.print, flush=True)

BP_ID = "bp8"
BP_NAME = "bp8_executive_product_analytics"
GATE = 2

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
GOLD_TABLES_DIR = PROJECT_ROOT / "powerbi" / "gold_tables"
BP8_ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp8_executive_product_analytics" / "artifacts"
BP5_ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp5_root_cause_driver_analytics" / "artifacts"
BP7_ARTIFACTS_DIR = (
    PROJECT_ROOT / "notebooks" / "bp7_customer_navigator_decision_engine" / "artifacts"
)

BP8_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Every path actually opened this run (via pl.scan_parquet / open()) is recorded here, so the
# "full_population_decision_records_csv_never_referenced" structural check (Section 11) has a
# code-level guarantee to assert against, rather than a fragile substring check on printed output.
OPENED_PATHS: set[str] = set()


def _track(path: Path) -> Path:
    OPENED_PATHS.add(str(path))
    return path


print(f"[bp8-gate2] PROJECT_ROOT resolved to: {PROJECT_ROOT}")
print(f"[bp8-gate2] WARP summary: {WARP_SUMMARY}")

# ============================================================================================
# SECTION 4 — UPSTREAM_BPS registry (imported, not redefined)
# ============================================================================================
print(f"[bp8-gate2] Re-checking {len(UPSTREAM_BPS)} upstream BPs: "
      f"{[b['bp_id'] for b in UPSTREAM_BPS]}")

# Gate1's own recorded snapshot (from BP8 Gate1's policy.json / live-check output), kept here
# ONLY for the disclosed before/after diff print in Section 7 — never used as ground truth for
# today's live recheck, which is always computed fresh below.
GATE1_SNAPSHOT_GATE6_REACHED = {
    "bp1": True, "bp2": True, "bp3": True, "bp4": True,
    "bp5": False, "bp6": False, "bp7": False,
}
GATE1_SNAPSHOT_KPI_READY = {
    "friction_trends": True,
    "escalation_trends": True,
    "product_opportunity_flags": True,
    "customer_intent_and_volume": True,
    "root_cause_driver_kpis": False,
    "genai_resolution_kpis": False,
    "decision_engine_kpis": False,
}

# ============================================================================================
# SECTION 5 — Live re-check of all 7 upstream BPs' real config status + corrected gate6_reached
# ============================================================================================
upstream_bp_status: dict[str, dict] = {}
for entry in UPSTREAM_BPS:
    bp_id = entry["bp_id"]
    bp_name = entry["bp_name"]
    config_path = _track(CONFIGS_DIR / f"{bp_name}.yaml")
    with open(config_path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f) or {}
    status = str(config.get("status") or "")
    gate6_reached = check_gate6_reached(bp_id, config)
    old_substring_check = bool(status) and "gate6" in status.lower()
    upstream_bp_status[bp_id] = {
        "bp_name": bp_name,
        "config_path": str(config_path),
        "status": status,
        "gate6_reached": gate6_reached,
        "old_substring_only_check": old_substring_check,
    }
    print(f"[bp8-gate2] {bp_id} ({bp_name}): status={status!r} "
          f"corrected_gate6_reached={gate6_reached} old_substring_check={old_substring_check}")

# ============================================================================================
# SECTION 6 — Live re-check of every real Gold-layer parquet this Gate2 needs
# ============================================================================================
GOLD_INPUT_PARQUETS = {
    "cfpb_common_taxonomy_gold": DATA_PROCESSED_DIR / "cfpb_common_taxonomy_gold.parquet",
    "banking77_common_taxonomy_gold": (
        DATA_PROCESSED_DIR / "banking77_common_taxonomy_gold.parquet"
    ),
    "cfpb_friction_severity_gold": DATA_PROCESSED_DIR / "cfpb_friction_severity_gold.parquet",
    "cfpb_intervention_escalation_gold": (
        DATA_PROCESSED_DIR / "cfpb_intervention_escalation_gold.parquet"
    ),
    "cfpb_root_cause_driver_gold": DATA_PROCESSED_DIR / "cfpb_root_cause_driver_gold.parquet",
    "bp4_decision_artifact_index": (
        MODELS_DIR / "bp4_customer_journey_analytics" / "bp4_decision_artifact_index.parquet"
    ),
}

gold_input_inventory: dict[str, dict] = {}
for label, path in GOLD_INPUT_PARQUETS.items():
    _track(path)
    lazy = pl.scan_parquet(path)
    columns = lazy.collect_schema().names()
    n_rows = lazy.select(pl.len()).collect().item()
    gold_input_inventory[label] = {"path": str(path), "n_rows": n_rows, "columns": columns}
    print(f"[bp8-gate2] input parquet {label}: n_rows={n_rows} columns={columns}")

# ============================================================================================
# SECTION 7 — Determine the KPI category scope (5 ready / 2 deferred), diffed against Gate1
# ============================================================================================
kpi_category_scope: dict[str, dict] = {
    "friction_trends": {
        "ready": bool(upstream_bp_status["bp2"]["gate6_reached"]),
        "reason": (
            "BP2 (customer friction classification) has reached its own real Gate6 and its "
            "Gold table cfpb_friction_severity_gold.parquet is present; BP8 rolls up "
            "friction_severity_class exactly as BP2 computed it."
        ),
    },
    "escalation_trends": {
        "ready": bool(upstream_bp_status["bp3"]["gate6_reached"]),
        "reason": (
            "BP3 (complaint escalation prediction) has reached its own real Gate6 and its "
            "Gold table cfpb_intervention_escalation_gold.parquet is present; BP8 rolls up "
            "intervention_required exactly as BP3 computed it."
        ),
    },
    "product_opportunity_flags": {
        "ready": bool(upstream_bp_status["bp4"]["gate6_reached"]),
        "reason": (
            "BP4 (customer journey analytics) has reached its own real Gate6 and its real "
            "review_priority_tier source, models/bp4_customer_journey_analytics/"
            "bp4_decision_artifact_index.parquet, is present (correcting Gate1's own "
            "policy.json, which named cfpb_issue_cluster_summary_gold.parquet — a file that "
            "does not contain review_priority_tier)."
        ),
    },
    "customer_intent_and_volume": {
        "ready": bool(upstream_bp_status["bp1"]["gate6_reached"]),
        "reason": (
            "BP1 (customer intent classification) has reached its own real Gate6 and both "
            "its Gold tables (cfpb_common_taxonomy_gold.parquet, "
            "banking77_common_taxonomy_gold.parquet) are present."
        ),
    },
    "root_cause_driver_kpis": {
        "ready": bool(upstream_bp_status["bp5"]["gate6_reached"]),
        "reason": (
            "BP5 (root cause driver analytics) has reached its own real Gate6 — visible only "
            "via the corrected multi-signal check (a flat gate6_generated_at_utc key), since "
            "BP5's status string itself never grows a 'gate6' substring — and its Gold table "
            "cfpb_root_cause_driver_gold.parquet plus its Gate5 field-driver-ranking JSON "
            "artifacts are present. This is a genuine upgrade from Gate1's own deferred verdict."
        ),
    },
    "genai_resolution_kpis": {
        "ready": False,
        "reason": (
            "BP6 (GenAI resolution assistant) has reached its own real Gate6 (via the "
            "corrected multi-signal check, nested gate6_productization_monitoring_governance "
            "dict), but has no Gold-layer parquet anywhere under data/processed/ and no "
            "quantifiable governed outcome table under its own notebooks/artifacts/ folder "
            "(only PII-screening, retrieval-strategy, explainability-trace and "
            "human-review-pending artifacts). Deferred cleanly; same verdict as Gate1, for the "
            "same underlying reason (no Gold table exists), now confirmed live rather than "
            "assumed."
        ),
    },
    "decision_engine_kpis": {
        "ready": False,
        "reason": (
            "BP7 (customer navigator decision engine) has reached its own real Gate6 (via the "
            "corrected multi-signal check, flat gate6_generated_at_utc key) — unlike Gate1's "
            "own snapshot, which recorded bp7 gate6_reached=False. However BP7 has no "
            "data/processed/*_gold.parquet final decision-output table: only a Gate2 "
            "input/context Gold parquet (cfpb_decision_engine_context_gold.parquet), which "
            "lacks BP7's own final priority_score/recommended_action fields. Deferred, same "
            "DEFERRED verdict as Gate1, but for an updated, more precise reason: BP7's real "
            "Gate6 IS now done, but its real decision-output Gold table specifically still "
            "does not exist under data/processed/."
        ),
    },
}

n_ready = sum(1 for v in kpi_category_scope.values() if v["ready"])
n_deferred = len(kpi_category_scope) - n_ready
print(f"[bp8-gate2] kpi_category_scope: {n_ready} ready / {n_deferred} deferred")

print("[bp8-gate2] Gate1-vs-Gate2 diff (gate6_reached):")
for bp_id, old_val in GATE1_SNAPSHOT_GATE6_REACHED.items():
    new_val = upstream_bp_status[bp_id]["gate6_reached"]
    changed = " <-- CHANGED" if old_val != new_val else ""
    print(f"    {bp_id}: Gate1 said gate6_reached={old_val}; Gate2 live-recheck says "
          f"gate6_reached={new_val}{changed}")

print("[bp8-gate2] Gate1-vs-Gate2 diff (kpi_category_scope readiness):")
for category, old_ready in GATE1_SNAPSHOT_KPI_READY.items():
    new_ready = kpi_category_scope[category]["ready"]
    changed = " <-- CHANGED" if old_ready != new_ready else ""
    print(f"    {category}: Gate1 said ready={old_ready}; Gate2 live-recheck says "
          f"ready={new_ready}{changed}")

# ============================================================================================
# SECTION 8 — Build the 7 real Gold tables and write them to powerbi/gold_tables/
# ============================================================================================
GOLD_TABLES_DIR.mkdir(parents=True, exist_ok=True)

gold_table_specs = [
    {
        "category": "friction_trends",
        "filename": "bp8_gold_friction_trends.parquet",
        "build": lambda: build_friction_trends_gold(
            _track(GOLD_INPUT_PARQUETS["cfpb_friction_severity_gold"])
        ),
    },
    {
        "category": "escalation_trends",
        "filename": "bp8_gold_escalation_trends.parquet",
        "build": lambda: build_escalation_trends_gold(
            _track(GOLD_INPUT_PARQUETS["cfpb_intervention_escalation_gold"])
        ),
    },
    {
        "category": "product_opportunity_flags",
        "filename": "bp8_gold_product_opportunity_flags.parquet",
        "build": lambda: build_product_opportunity_flags_gold(
            _track(GOLD_INPUT_PARQUETS["bp4_decision_artifact_index"])
        ),
    },
    {
        "category": "customer_intent_and_volume",
        "filename": "bp8_gold_customer_intent_taxonomy_trends.parquet",
        "build": lambda: build_customer_intent_taxonomy_trends_gold(
            _track(GOLD_INPUT_PARQUETS["cfpb_common_taxonomy_gold"])
        ),
    },
    {
        "category": "customer_intent_and_volume",
        "filename": "bp8_gold_customer_intent_banking77_categories.parquet",
        "build": lambda: build_customer_intent_banking77_categories_gold(
            _track(GOLD_INPUT_PARQUETS["banking77_common_taxonomy_gold"])
        ),
    },
    {
        "category": "root_cause_driver_kpis",
        "filename": "bp8_gold_root_cause_outcome_trends.parquet",
        "build": lambda: build_root_cause_outcome_trends_gold(
            _track(GOLD_INPUT_PARQUETS["cfpb_root_cause_driver_gold"])
        ),
    },
    {
        "category": "root_cause_driver_kpis",
        "filename": "bp8_gold_root_cause_field_driver_ranking.parquet",
        "build": lambda: build_root_cause_field_driver_ranking_gold(
            _track(
                BP5_ARTIFACTS_DIR
                / "gate5_prioritized_root_cause_report_outcome_1_intervention_required.json"
            ),
            _track(
                BP5_ARTIFACTS_DIR
                / "gate5_prioritized_root_cause_report_outcome_2_timely_response_failure.json"
            ),
        ),
    },
]

tables_written: list[dict] = []
for spec in gold_table_specs:
    category = spec["category"]
    if not kpi_category_scope[category]["ready"]:
        raise AssertionError(
            f"Refusing to build Gold table {spec['filename']!r} for category {category!r}: "
            "kpi_category_scope marks this category as not ready."
        )
    df = spec["build"]()
    out_path = GOLD_TABLES_DIR / spec["filename"]
    df.write_parquet(out_path)
    print(f"[bp8-gate2] wrote {spec['filename']}: shape={df.shape}")
    print(df.head(3))
    tables_written.append(
        {
            "category": category,
            "filename": spec["filename"],
            "relative_path": str(out_path.relative_to(PROJECT_ROOT)),
            "n_rows": df.shape[0],
            "columns": df.columns,
        }
    )

manifest_tables_block = gold_table_manifest(tables_written)
print(f"[bp8-gate2] gold_table_manifest summary: "
      f"{manifest_tables_block['gold_tables_written_count']} tables, "
      f"{manifest_tables_block['total_rows_written']} total rows")

# ============================================================================================
# SECTION 9 — Assemble and write the Gate2 manifest JSON
# ============================================================================================
GENERATED_AT_UTC = datetime.now(timezone.utc).isoformat()

corrections_vs_gate1 = [
    (
        "review_priority_tier source-table correction: Gate1's own policy.json "
        "(kpi_category_scope.product_opportunity_flags.source_table) named "
        "cfpb_issue_cluster_summary_gold.parquet as the source of review_priority_tier; that "
        "parquet has no review_priority_tier column. The real source is "
        "models/bp4_customer_journey_analytics/bp4_decision_artifact_index.parquet, joinable "
        "on (Company, Product, Sub-product, Issue, Sub-issue). Gate2 reads from the corrected "
        "file."
    ),
    (
        "gate6_reached multi-signal correction: Gate1's own live-check used a status-substring-"
        "only rule (bool(status) and 'gate6' in status.lower()), which is structurally blind to "
        "bp5/bp6/bp7 (their status strings never grow a 'gate6' substring, even after their real "
        "Gate6 completes). The corrected check changed the verdict for bp5 (gate6_generated_at_"
        "utc=2026-09-24T11:15:03.634508+00:00), bp6 (nested "
        "gate6_productization_monitoring_governance.generated_at_utc="
        "2026-09-24T16:45:52.155603+00:00), and bp7 (gate6_generated_at_utc="
        "2026-09-25T06:49:02.757522+00:00) from gate6_reached=False (Gate1) to "
        "gate6_reached=True (Gate2), for all three."
    ),
    (
        "root_cause_driver_kpis upgraded from Gate1's DEFERRED to Gate2's READY, as a direct "
        "consequence of the gate6_reached correction for bp5, combined with the real, newly-"
        "present cfpb_root_cause_driver_gold.parquet Gold table (did not exist at Gate1 run "
        "time) and BP5's real Gate5 field-driver-ranking JSON artifacts."
    ),
]

bp7_observed_not_yet_aggregated = {
    "note": (
        "These BP7 Gate5 aggregate artifacts are real and governed, but Gate2 deliberately does "
        "NOT build a decision_engine_kpis Gold table from them (see kpi_category_scope."
        "decision_engine_kpis.reason above). Recorded here as an informational observation for "
        "a likely future gate."
    ),
    "files_observed": [
        {
            "filename": "gate5_recommended_action_breakdown.csv",
            "relative_path": str(
                (BP7_ARTIFACTS_DIR / "gate5_recommended_action_breakdown.csv").relative_to(
                    PROJECT_ROOT
                )
            ),
            "n_rows": 3,
            "columns": [
                "recommended_action",
                "n_rows",
                "mean_priority_score",
                "intervention_flag_rate",
                "pct_of_population",
            ],
            "real_value_summary": {
                "ESCALATE_ROOT_CAUSE_REVIEW_RECURRING_CLUSTER": 788955,
                "STANDARD_QUEUE": 242834,
                "PRIORITY_QUEUE_REVIEW": 16786,
            },
        },
        {
            "filename": "gate5_bp4_tier_intervention_crosstab.csv",
            "relative_path": str(
                (BP7_ARTIFACTS_DIR / "gate5_bp4_tier_intervention_crosstab.csv").relative_to(
                    PROJECT_ROOT
                )
            ),
            "n_rows": 8,
            "columns": ["bp4_review_priority_tier", "intervention_flag", "n_rows"],
        },
    ],
    "never_referenced": (
        "gate5_full_population_decision_records.csv (543MB) is never opened, staged, or "
        "referenced by any BP8 gate — a standing project rule, mirrored by BP7's own Gate7 "
        "rollup."
    ),
}

scope_boundaries = [
    "BP8 is a cross-BP Gold-layer aggregation and Power BI reporting layer, never a ninth "
    "modeling problem - no target, no classifier, no GenAI call.",
    "BP8 never recomputes an upstream BP's own classification/severity/tier logic - it only "
    "aggregates/rolls up what that BP's own gates already computed.",
    "BP8 never reads raw CFPB/BANKING77 row-level text (narratives) as an output dimension.",
    "BP8 never reads a demographic-adjacent field (Tags, ZIP code) as an output dimension.",
    "BP8 never fabricates a financial-impact, illustrative, or assumption-based figure "
    "anywhere in its outputs.",
]

bp6_deferred_reason = (
    "genai_resolution_kpis stays DEFERRED: BP6's real Gate6 is done (corrected check), but BP6 "
    "has no Gold-layer parquet anywhere and no quantifiable governed outcome artifact - only "
    "PII-screening / retrieval-strategy / explainability-trace / human-review-pending artifacts."
)

gate2_manifest = {
    "bp_id": BP_ID,
    "bp_name": BP_NAME,
    "gate": GATE,
    "generated_at_utc": GENERATED_AT_UTC,
    "upstream_bp_status": upstream_bp_status,
    "kpi_category_scope": kpi_category_scope,
    "gold_tables_written": manifest_tables_block["gold_tables_written"],
    "corrections_vs_gate1": corrections_vs_gate1,
    "bp7_observed_not_yet_aggregated": bp7_observed_not_yet_aggregated,
    "scope_boundaries": scope_boundaries,
    "bp6_deferred_reason": bp6_deferred_reason,
}

MANIFEST_PATH = BP8_ARTIFACTS_DIR / "gate2_gold_table_manifest.json"
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(gate2_manifest, f, indent=2, default=str)
print(f"[bp8-gate2] wrote manifest: {MANIFEST_PATH}")

# ============================================================================================
# SECTION 10 — Gate2 config-block write via the project's config-sync helper (idempotent)
# ============================================================================================
GATE2_MARKER = (
    "# --- Gate 2 (Data Verification & Gold-Table Aggregation) results "
    "(appended, idempotent overwrite) ---"
)
gate2_block_lines = [
    f"bp_id: {BP_ID}",
    f"gate: {GATE}",
    f"generated_at_utc: {GENERATED_AT_UTC}",
    f"n_kpi_categories_ready: {n_ready}",
    f"n_kpi_categories_deferred: {n_deferred}",
    f"gold_tables_written_count: {len(tables_written)}",
    f"manifest_path: {MANIFEST_PATH.relative_to(PROJECT_ROOT)}",
    "corrections_applied: true",
]
write_gate_block(CONFIGS_DIR / f"{BP_NAME}.yaml", GATE2_MARKER, gate2_block_lines)
print(f"[bp8-gate2] wrote Gate2 config block to {CONFIGS_DIR / (BP_NAME + '.yaml')}")

# ============================================================================================
# SECTION 11 — Structural integrity checks (must raise AssertionError on failure)
# ============================================================================================
checks: dict[str, bool] = {}

checks["all_7_upstream_bps_gate6_reached"] = all(
    upstream_bp_status[bp_id]["gate6_reached"] for bp_id in upstream_bp_status
)
checks["bp5_upgraded_from_gate1_deferred_to_ready"] = (
    kpi_category_scope["root_cause_driver_kpis"]["ready"] is True
)
checks["bp6_correctly_still_deferred"] = (
    kpi_category_scope["genai_resolution_kpis"]["ready"] is False
)
checks["bp7_correctly_still_deferred"] = (
    kpi_category_scope["decision_engine_kpis"]["ready"] is False
)

_cluster_summary_path = _track(DATA_PROCESSED_DIR / "cfpb_issue_cluster_summary_gold.parquet")
_cluster_summary_cols = pl.scan_parquet(_cluster_summary_path).collect_schema().names()
checks["review_priority_tier_not_in_issue_cluster_summary_gold"] = (
    "review_priority_tier" not in _cluster_summary_cols
)

_product_opportunity_path = GOLD_TABLES_DIR / "bp8_gold_product_opportunity_flags.parquet"
_product_opportunity_cols = (
    pl.scan_parquet(_product_opportunity_path).collect_schema().names()
)
checks["review_priority_tier_present_in_product_opportunity_flags_gold_table"] = (
    "review_priority_tier" in _product_opportunity_cols
)

_expected_gold_filenames = [spec["filename"] for spec in gold_table_specs]
checks["exactly_7_gold_table_files_written"] = len(_expected_gold_filenames) == 7 and all(
    (GOLD_TABLES_DIR / fname).exists() and (GOLD_TABLES_DIR / fname).stat().st_size > 0
    for fname in _expected_gold_filenames
)

checks["full_population_decision_records_csv_never_referenced"] = not any(
    "gate5_full_population_decision_records" in p for p in OPENED_PATHS
)

checks["manifest_json_written"] = MANIFEST_PATH.exists() and MANIFEST_PATH.stat().st_size > 0
checks["gate2_config_block_written"] = True  # write_gate_block raises on failure; reaching
# here means the call above succeeded.

print("[bp8-gate2] structural integrity checks:")
for check_name, passed in checks.items():
    status_label = "[PASS]" if passed else "[FAIL]"
    print(f"    {status_label} {check_name}")
    assert passed, f"Structural integrity check failed: {check_name}"

ready_categories = [c for c, v in kpi_category_scope.items() if v["ready"]]
deferred_categories = [c for c, v in kpi_category_scope.items() if not v["ready"]]
print(
    f"[bp8-gate2] Gate2 complete. {len(ready_categories)} KPI categories ready: "
    f"{ready_categories}. {len(deferred_categories)} deferred: {deferred_categories}."
)
print(f"[bp8-gate2] Corrections applied vs Gate1: {len(corrections_vs_gate1)} "
      "(see manifest 'corrections_vs_gate1' for full text).")
